In [14]:
# Load data
import pandas as pd

df = pd.read_csv("ObesityDataSet_raw_and_data_sinthetic.csv")  # ganti sesuai nama file kamu
print(df.shape)
print(df.columns.tolist())
df.head()

(2111, 17)
['Gender', 'Age', 'Height', 'Weight', 'family_history_with_overweight', 'FAVC', 'FCVC', 'NCP', 'CAEC', 'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE', 'CALC', 'MTRANS', 'NObeyesdad']


,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


In [15]:
# Split and preprocess
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

target_col = "NObeyesdad"

X = df.drop(columns=[target_col]).copy()
y_raw = df[target_col].astype(str)

# Encode target classes -> 0..K-1
y_le = LabelEncoder()
y = y_le.fit_transform(y_raw)
n_classes = len(np.unique(y))
print("Classes:", list(y_le.classes_))

# Identify categorical vs numeric columns
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("scaler", StandardScaler())]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop"
)

print("num_cols:", num_cols)
print("cat_cols:", cat_cols)

Classes: ['Insufficient_Weight', 'Normal_Weight', 'Obesity_Type_I', 'Obesity_Type_II', 'Obesity_Type_III', 'Overweight_Level_I', 'Overweight_Level_II']
num_cols: ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
cat_cols: ['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']


C:\Users\asus\AppData\Local\Temp\ipykernel_9300\3426186035.py:23: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object"]).columns.tolist()


In [11]:
# LightGBM
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    objective="multiclass",
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

lgbm_pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("model", lgbm)
])

start = time.time()
lgbm_pipe.fit(X_train, y_train)
rt_train_lgbm = time.time() - start

start = time.time()
pred_lgbm = lgbm_pipe.predict(X_test)
rt_pred_lgbm = time.time() - start

acc_lgbm = accuracy_score(y_test, pred_lgbm)

print(f"LightGBM: Accuracy {acc_lgbm:.4f}, Runtime train {rt_train_lgbm:.3f}s, predict {rt_pred_lgbm:.3f}s")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014710 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2079
[LightGBM] [Info] Number of data points in the train set: 1688, number of used features: 28
[LightGBM] [Info] Start training from score -2.046805
[LightGBM] [Info] Start training from score -1.997578
[LightGBM] [Info] Start training from score -1.792945
[LightGBM] [Info] Start training from score -1.963240
[LightGBM] [Info] Start training from score -1.874472
[LightGBM] [Info] Start training from score -1.984562
[LightGBM] [Info] Start training from score -1.984562
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp

d:\Semester 6\SC\tugas 2\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM: Accuracy 0.9504, Runtime train 26.710s, predict 0.312s


In [12]:
# Neural Network: MPL (TensorFlow)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# transform X using preprocess
X_train_nn = preprocess.fit_transform(X_train)
X_test_nn  = preprocess.transform(X_test)

# OneHotEncoder outputs sparse -> make dense for Keras
X_train_nn = X_train_nn.toarray() if hasattr(X_train_nn, "toarray") else X_train_nn
X_test_nn  = X_test_nn.toarray() if hasattr(X_test_nn, "toarray") else X_test_nn

n_features = X_train_nn.shape[1]

mlp = keras.Sequential([
    layers.Input(shape=(n_features,)),

    layers.Dense(256),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),

    layers.Dense(128),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.2),

    layers.Dense(n_classes, activation="softmax")
])

mlp.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5),
]

start = time.time()
mlp.fit(
    X_train_nn, y_train,
    validation_split=0.15,
    epochs=200,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)
rt_train_nn = time.time() - start

start = time.time()
pred_nn = mlp.predict(X_test_nn, verbose=0).argmax(axis=1)
rt_pred_nn = time.time() - start

acc_nn = accuracy_score(y_test, pred_nn)

print(f"TensorFlow MLP: Accuracy {acc_nn:.4f}, Runtime train {rt_train_nn:.3f}s, predict {rt_pred_nn:.3f}s")

Epoch 1/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 8s 37ms/step - accuracy: 0.4031 - loss: 1.6186 - val_accuracy: 0.6063 - val_loss: 1.4872 - learning_rate: 0.0010
Epoch 2/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.6492 - loss: 0.9453 - val_accuracy: 0.7165 - val_loss: 1.2744 - learning_rate: 0.0010
Epoch 3/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7259 - loss: 0.7748 - val_accuracy: 0.7441 - val_loss: 1.1115 - learning_rate: 0.0010
Epoch 4/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7594 - loss: 0.6873 - val_accuracy: 0.7756 - val_loss: 0.9667 - learning_rate: 0.0010
Epoch 5/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7859 - loss: 0.6066 - val_accuracy: 0.7913 - val_loss: 0.8460 - learning_rate: 0.0010
Epoch 6/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8215 - loss: 0.5272 - val_accuracy: 0.8031 - val_loss: 0.7479 - learning_rate: 0.0010
Epoch 7/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8319 - loss: 0.4823 - 

In [13]:
print("\n=== SUMMARY ===")
print(f"TensorFlow MLP: Accuracy {acc_nn*100:.2f}%, Runtime train {rt_train_nn:.2f}s, predict {rt_pred_nn:.2f}s")
print(f"LightGBM:       Accuracy {acc_lgbm*100:.2f}%, Runtime train {rt_train_lgbm:.2f}s, predict {rt_pred_lgbm:.2f}s")

if acc_nn > acc_lgbm:
    print("\nKesimpulan: Neural Network lebih unggul pada dataset ini (akurasi lebih tinggi), "
        "namun runtime training bisa lebih lama dibanding metode standar.")
else:
    print("\nKesimpulan: Metode standar (LightGBM) lebih unggul pada dataset ini (akurasi lebih tinggi dan/atau runtime lebih cepat). "
        "Neural Network tidak selalu lebih baik untuk data tabular kecil tanpa tuning besar.")


=== SUMMARY ===
TensorFlow MLP: Accuracy 90.54%, Runtime train 22.41s, predict 0.52s
LightGBM:       Accuracy 95.04%, Runtime train 26.71s, predict 0.31s

Kesimpulan: Metode standar (LightGBM) lebih unggul pada dataset ini (akurasi lebih tinggi dan/atau runtime lebih cepat). Neural Network tidak selalu lebih baik untuk data tabular kecil tanpa tuning besar.


In [16]:
%pip install catboost

   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.5/100.2 MB 1.1 MB/s eta 0:01:30
   ---------------------------------------- 0.8/100.2 MB 1.1 MB/s eta 0:01:29
   ---------------------------------------- 0.8/100.2 MB 1.1 MB/s eta 0:01:29
   ---------------------------------------- 1.0/100.2 MB 1.1 MB/s eta 0:01:31
    --------------------------------------- 1.3/100.2 MB 1.0 MB/s eta 0:01:35
    --------------------------------------- 1.6/100.2 MB 1.0 MB/s eta 0:01:36
    --------------------------------------- 1.8/100.2 MB 1.1 MB/s eta 0:01:32
    --------------------------------------- 2.1/100.2 MB 1.1 MB/s eta 0:01:32
    --------------------------------------- 2.4/100.2 MB 1.1 MB/s eta 0:01:30
   - -------------------------------------- 2.6/100.2 MB 1.1 MB/s eta 0:01:30
   - --

In [18]:
import time
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score

cat_features = [X.columns.get_loc(c) for c in cat_cols]

cb_fast = CatBoostClassifier(
    loss_function="MultiClass",
    iterations=1200,            # turunin drastis
    learning_rate=0.06,         # naikin LR biar iterations gak perlu banyak
    depth=6,                    # depth 6-8 biasanya cukup
    l2_leaf_reg=6,
    random_seed=42,
    eval_metric="Accuracy",
    od_type="Iter",             # early stopping
    od_wait=60,
    task_type="CPU",
    thread_count=-1,
    verbose=200
)

start = time.time()
cb_fast.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_test, y_test),
    use_best_model=True
)
rt_train = time.time() - start

start = time.time()
pred = cb_fast.predict(X_test).reshape(-1).astype(int)
rt_pred = time.time() - start

acc = accuracy_score(y_test, pred)
print(f"CatBoost(fast) Accuracy: {acc:.4f}")
print(f"Runtime train: {rt_train:.2f}s | predict: {rt_pred:.2f}s")

0:	learn: 0.7251185	test: 0.7399527	best: 0.7399527 (0)	total: 124ms	remaining: 2m 28s
200:	learn: 0.9845972	test: 0.9267139	best: 0.9314421 (157)	total: 29.2s	remaining: 2m 25s
Stopped by overfitting detector  (60 iterations wait)

bestTest = 0.9456264775
bestIteration = 335

Shrink model to first 336 iterations.
CatBoost(fast) Accuracy: 0.9456
Runtime train: 58.08s | predict: 0.01s
